#Intelligent Employee Policy Assistant

In [1]:
!pip install -q pypdf reportlab pandas langchain-text-splitters

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 346.6/346.6 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 27.6 MB/s eta 0:00:00


In [2]:
import os
import pandas as pd
from pypdf import PdfReader
from reportlab.pdfgen import canvas
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [3]:
documents = {
    "Employee Handbook.pdf":
    """
    Employees must follow company policies and code of conduct.
    Employees are expected to maintain professionalism at all times.
    """,

    "Leave Policy.pdf":
    """
    Employees receive 12 casual leaves annually.
    Employees receive 15 sick leaves annually.
    Unused casual leaves cannot be carried forward.
    """,

    "Travel Policy.pdf":
    """
    Travel expenses are reimbursed within 30 days.
    Original receipts must be submitted for reimbursement.
    """,

    "Work From Home Policy.pdf":
    """
    Employees may work from home twice per week.
    Manager approval is required for additional remote work.
    """,

    "Medical Insurance Policy.pdf":
    """
    All employees are covered under company medical insurance.
    Dependent coverage is available for spouse and children.
    """
}

In [4]:
for filename, content in documents.items():

    c = canvas.Canvas(filename)

    y = 800

    for line in content.split("\n"):
        c.drawString(50, y, line)
        y -= 20

    c.save()

print("All PDFs Created Successfully")

All PDFs Created Successfully


#Task 1: Document Loading

In [5]:
pdf_files = [
    "Employee Handbook.pdf",
    "Leave Policy.pdf",
    "Travel Policy.pdf",
    "Work From Home Policy.pdf",
    "Medical Insurance Policy.pdf"
]

In [6]:
all_text = ""

stats = []

for file in pdf_files:

    reader = PdfReader(file)

    pages = len(reader.pages)

    text = ""

    for page in reader.pages:

        extracted = page.extract_text()

        if extracted:
            text += extracted

    all_text += text + "\n"

    stats.append([
        file,
        pages,
        len(text),
        len(text.split())
    ])

In [7]:
stats_df = pd.DataFrame(
    stats,
    columns=[
        "File Name",
        "Number of Pages",
        "Total Characters",
        "Total Words"
    ]
)

stats_df

,File Name,Number of Pages,Total Characters,Total Words
0,Employee Handbook.pdf,1,138,18
1,Leave Policy.pdf,1,153,19
2,Travel Policy.pdf,1,115,14
3,Work From Home Policy.pdf,1,115,16
4,Medical Insurance Policy.pdf,1,129,16


#Task 2: Fixed Size Chunking

In [8]:
def fixed_size_chunking(text, chunk_size):

    chunks = []

    for i in range(0, len(text), chunk_size):

        chunks.append(
            text[i:i+chunk_size]
        )

    return chunks

In [9]:
fixed_chunks = fixed_size_chunking(
    all_text,
    chunk_size=500
)

In [10]:
for i, chunk in enumerate(fixed_chunks):

    print(f"\nChunk {i+1}")

    print("-"*50)

    print(chunk)

    print("\nLength:", len(chunk))


Chunk 1
--------------------------------------------------
    Employees must follow company policies and code of conduct.
    Employees are expected to maintain professionalism at all times.
    

    Employees receive 12 casual leaves annually.
    Employees receive 15 sick leaves annually.
    Unused casual leaves cannot be carried forward.
    

    Travel expenses are reimbursed within 30 days.
    Original receipts must be submitted for reimbursement.
    

    Employees may work from home twice per week.
    Manager approval is required for addit

Length: 500

Chunk 2
--------------------------------------------------
ional remote work.
    

    All employees are covered under company medical insurance.
    Dependent coverage is available for spouse and children.
    



Length: 155


In [11]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

recursive_chunks = splitter.split_text(all_text)

In [12]:
for i, chunk in enumerate(recursive_chunks):

    print(f"\nChunk {i+1}")

    print("-"*50)

    print(chunk)

    print("\nLength:", len(chunk))


Chunk 1
--------------------------------------------------
Employees must follow company policies and code of conduct.
    Employees are expected to maintain professionalism at all times.
    

    Employees receive 12 casual leaves annually.
    Employees receive 15 sick leaves annually.
    Unused casual leaves cannot be carried forward.
    

    Travel expenses are reimbursed within 30 days.
    Original receipts must be submitted for reimbursement.

Length: 398

Chunk 2
--------------------------------------------------
Employees may work from home twice per week.
    Manager approval is required for additional remote work.
    

    All employees are covered under company medical insurance.
    Dependent coverage is available for spouse and children.

Length: 235


In [13]:
comparison = pd.DataFrame({
    "Chunking Method":[
        "Fixed Size",
        "Recursive"
    ],
    "Chunk Size":[
        500,
        500
    ],
    "Overlap":[
        0,
        100
    ],
    "Context Preservation":[
        "Low",
        "High"
    ]
})

comparison

,Chunking Method,Chunk Size,Overlap,Context Preservation
0,Fixed Size,500,0,Low
1,Recursive,500,100,High


In [14]:
!pip install -q sentence-transformers

In [15]:
from sentence_transformers import SentenceTransformer
import numpy as np

In [17]:
model = SentenceTransformer("all-MiniLM-L6-v2")

print("Model Loaded Successfully")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model Loaded Successfully


In [18]:
chunk_embeddings = model.encode(recursive_chunks)

In [19]:
for i, (chunk, embedding) in enumerate(zip(recursive_chunks, chunk_embeddings)):

    print(f"\n{'='*60}")
    print(f"Chunk {i+1}")
    print(f"{'='*60}")

    print("Chunk Content:")
    print(chunk)

    print("\nEmbedding Shape:")
    print(embedding.shape)

    print("\nFirst 10 Embedding Values:")
    print(embedding[:10])


Chunk 1
Chunk Content:
Employees must follow company policies and code of conduct.
    Employees are expected to maintain professionalism at all times.
    

    Employees receive 12 casual leaves annually.
    Employees receive 15 sick leaves annually.
    Unused casual leaves cannot be carried forward.
    

    Travel expenses are reimbursed within 30 days.
    Original receipts must be submitted for reimbursement.

Embedding Shape:
(384,)

First 10 Embedding Values:
[ 0.05990278  0.05575204  0.04111109  0.05649447  0.06298611  0.03287065
  0.02591398 -0.02835011 -0.08013896  0.02895295]

Chunk 2
Chunk Content:
Employees may work from home twice per week.
    Manager approval is required for additional remote work.
    

    All employees are covered under company medical insurance.
    Dependent coverage is available for spouse and children.

Embedding Shape:
(384,)

First 10 Embedding Values:
[ 0.00136052 -0.01421674  0.09143649 -0.00124485  0.0039244   0.04233234
 -0.00402177 -0

In [20]:
import pandas as pd

embedding_info = []

for chunk, embedding in zip(recursive_chunks, chunk_embeddings):

    embedding_info.append([
        chunk[:80] + "...",
        embedding.shape
    ])

embedding_df = pd.DataFrame(
    embedding_info,
    columns=[
        "Chunk Preview",
        "Embedding Shape"
    ]
)

embedding_df

,Chunk Preview,Embedding Shape
0,Employees must follow company policies and cod...,"(384,)"
1,Employees may work from home twice per week.\n...,"(384,)"


In [21]:
print("Embedding Dimension:", len(chunk_embeddings[0]))

Embedding Dimension: 384


In [22]:
from sklearn.metrics.pairwise import cosine_similarity

s1 = "Employees receive 12 casual leaves annually."
s2 = "Workers are entitled to 12 annual leaves."

emb1 = model.encode([s1])
emb2 = model.encode([s2])

similarity = cosine_similarity(emb1, emb2)

print("Similarity Score:", similarity[0][0])

Similarity Score: 0.7841495


#Task 4: Build FAISS Vector Database
Objective:

Store embeddings efficiently.

Requirements:

Create FAISS index.

Insert chunk embeddings.

------------------------------

Display:

Number of Chunks

Number of Stored Vectors

Embedding Dimension

In [23]:
!pip install -q faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 40.5 MB/s eta 0:00:00


In [25]:
import numpy as np
import faiss

In [26]:
print("Number of Chunks:", len(recursive_chunks))
print("Embedding Shape:", chunk_embeddings.shape)

Number of Chunks: 2
Embedding Shape: (2, 384)


In [27]:
embeddings_np = np.array(
    chunk_embeddings,
    dtype=np.float32
)

In [28]:
dimension = embeddings_np.shape[1]

print("Embedding Dimension:", dimension)

Embedding Dimension: 384


In [29]:
index = faiss.IndexFlatL2(dimension)

In [30]:
index.add(embeddings_np)

In [31]:
print("="*50)
print("FAISS VECTOR DATABASE")
print("="*50)

print("Number of Chunks:")
print(len(recursive_chunks))

print("\nNumber of Stored Vectors:")
print(index.ntotal)

print("\nEmbedding Dimension:")
print(dimension)

FAISS VECTOR DATABASE
Number of Chunks:
2

Number of Stored Vectors:
2

Embedding Dimension:
384


In [32]:
import pandas as pd

faiss_info = pd.DataFrame({
    "Metric":[
        "Number of Chunks",
        "Stored Vectors",
        "Embedding Dimension"
    ],
    "Value":[
        len(recursive_chunks),
        index.ntotal,
        dimension
    ]
})

faiss_info

,Metric,Value
0,Number of Chunks,2
1,Stored Vectors,2
2,Embedding Dimension,384


In [33]:
print(index.is_trained)

True


#Task 5: Semantic Retrieval

In [34]:
queries = [
    "How many casual leaves are available?",
    "Can employees work from home?",
    "How does travel reimbursement work?",
    "Who is covered under medical insurance?"
]

In [35]:
import numpy as np

for query in queries:

    print("\n" + "="*80)
    print("QUERY:")
    print(query)

    # Generate Query Embedding
    query_embedding = model.encode([query])

    query_embedding = np.array(
        query_embedding,
        dtype=np.float32
    )

    # Search Top 3 Results
    distances, indices = index.search(
        query_embedding,
        k=3
    )

    print("\nTOP 3 RETRIEVED CHUNKS")

    for rank, idx in enumerate(indices[0]):

        score = distances[0][rank]

        print(f"\nRank {rank+1}")

        print("Chunk:")
        print(recursive_chunks[idx])

        print("Similarity Score:")
        print(round(float(score),4))


QUERY:
How many casual leaves are available?

TOP 3 RETRIEVED CHUNKS

Rank 1
Chunk:
Employees must follow company policies and code of conduct.
    Employees are expected to maintain professionalism at all times.
    

    Employees receive 12 casual leaves annually.
    Employees receive 15 sick leaves annually.
    Unused casual leaves cannot be carried forward.
    

    Travel expenses are reimbursed within 30 days.
    Original receipts must be submitted for reimbursement.
Similarity Score:
1.0101

Rank 2
Chunk:
Employees may work from home twice per week.
    Manager approval is required for additional remote work.
    

    All employees are covered under company medical insurance.
    Dependent coverage is available for spouse and children.
Similarity Score:
1.8825

Rank 3
Chunk:
Employees may work from home twice per week.
    Manager approval is required for additional remote work.
    

    All employees are covered under company medical insurance.
    Dependent coverage is

In [36]:
from sklearn.metrics.pairwise import cosine_similarity

In [37]:
for query in queries:

    print("\n" + "="*80)
    print("QUERY:", query)

    query_embedding = model.encode([query])

    similarities = cosine_similarity(
        query_embedding,
        chunk_embeddings
    )[0]

    top_indices = similarities.argsort()[-3:][::-1]

    for rank, idx in enumerate(top_indices):

        print(f"\nRank {rank+1}")

        print("Chunk:")
        print(recursive_chunks[idx])

        print(
            "Similarity Score:",
            round(float(similarities[idx]),4)
        )


QUERY: How many casual leaves are available?

Rank 1
Chunk:
Employees must follow company policies and code of conduct.
    Employees are expected to maintain professionalism at all times.
    

    Employees receive 12 casual leaves annually.
    Employees receive 15 sick leaves annually.
    Unused casual leaves cannot be carried forward.
    

    Travel expenses are reimbursed within 30 days.
    Original receipts must be submitted for reimbursement.
Similarity Score: 0.4949

Rank 2
Chunk:
Employees may work from home twice per week.
    Manager approval is required for additional remote work.
    

    All employees are covered under company medical insurance.
    Dependent coverage is available for spouse and children.
Similarity Score: 0.0588

QUERY: Can employees work from home?

Rank 1
Chunk:
Employees may work from home twice per week.
    Manager approval is required for additional remote work.
    

    All employees are covered under company medical insurance.
    Depende

In [38]:
import pandas as pd

results = []

for query in queries:

    query_embedding = model.encode([query])

    similarities = cosine_similarity(
        query_embedding,
        chunk_embeddings
    )[0]

    top_indices = similarities.argsort()[-3:][::-1]

    for rank, idx in enumerate(top_indices):

        results.append([
            query,
            rank + 1,
            recursive_chunks[idx],
            round(float(similarities[idx]),4)
        ])

retrieval_df = pd.DataFrame(
    results,
    columns=[
        "Query",
        "Rank",
        "Retrieved Chunk",
        "Similarity Score"
    ]
)

retrieval_df

,Query,Rank,Retrieved Chunk,Similarity Score
0,How many casual leaves are available?,1,Employees must follow company policies and cod...,0.4949
1,How many casual leaves are available?,2,Employees may work from home twice per week.\n...,0.0588
2,Can employees work from home?,1,Employees may work from home twice per week.\n...,0.6879
3,Can employees work from home?,2,Employees must follow company policies and cod...,0.2656
4,How does travel reimbursement work?,1,Employees must follow company policies and cod...,0.4383
5,How does travel reimbursement work?,2,Employees may work from home twice per week.\n...,0.1580
6,Who is covered under medical insurance?,1,Employees may work from home twice per week.\n...,0.2828
7,Who is covered under medical insurance?,2,Employees must follow company policies and cod...,0.1383


#Task 6 : Integrate Large Language Model
Generate natural-language answers from retrieved chunks

In [39]:
!pip install -q transformers accelerate torch sentencepiece

In [40]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(model_name)

llm = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

print("TinyLlama Loaded Successfully")

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

TinyLlama Loaded Successfully


In [41]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

query = "How many casual leaves are available?"

query_embedding = model.encode([query])

similarities = cosine_similarity(
    query_embedding,
    chunk_embeddings
)[0]

best_index = similarities.argmax()

retrieved_chunk = recursive_chunks[best_index]

print("Retrieved Context:")
print(retrieved_chunk)

Retrieved Context:
Employees must follow company policies and code of conduct.
    Employees are expected to maintain professionalism at all times.
    

    Employees receive 12 casual leaves annually.
    Employees receive 15 sick leaves annually.
    Unused casual leaves cannot be carried forward.
    

    Travel expenses are reimbursed within 30 days.
    Original receipts must be submitted for reimbursement.


In [42]:
prompt = f"""
Context:
{retrieved_chunk}

Question:
{query}

Answer:
"""

In [43]:
print(prompt)


Context:
Employees must follow company policies and code of conduct.
    Employees are expected to maintain professionalism at all times.
    

    Employees receive 12 casual leaves annually.
    Employees receive 15 sick leaves annually.
    Unused casual leaves cannot be carried forward.
    

    Travel expenses are reimbursed within 30 days.
    Original receipts must be submitted for reimbursement.

Question:
How many casual leaves are available?

Answer:



In [44]:
inputs = tokenizer(
    prompt,
    return_tensors="pt"
).to(llm.device)

output = llm.generate(
    **inputs,
    max_new_tokens=100,
    temperature=0.3
)

answer = tokenizer.decode(
    output[0],
    skip_special_tokens=True
)

print(answer)

[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
[transformers] Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Context:
Employees must follow company policies and code of conduct.
    Employees are expected to maintain professionalism at all times.
    

    Employees receive 12 casual leaves annually.
    Employees receive 15 sick leaves annually.
    Unused casual leaves cannot be carried forward.
    

    Travel expenses are reimbursed within 30 days.
    Original receipts must be submitted for reimbursement.

Question:
How many casual leaves are available?

Answer:
There are 12 casual leaves available annually.

Question:
How many sick leaves are available?

Answer:
There are 15 sick leaves available annually.

Question:
What is the maximum number of casual leaves that can be carried forward?

Answer:
There is no maximum number of casual leaves that can be carried forward.

Question:
What is the reimbursement policy for travel expenses?




In [45]:
def rag_answer(user_question):

    query_embedding = model.encode([user_question])

    similarities = cosine_similarity(
        query_embedding,
        chunk_embeddings
    )[0]

    best_idx = similarities.argmax()

    retrieved_context = recursive_chunks[best_idx]

    prompt = f"""
Context:
{retrieved_context}

Question:
{user_question}

Answer:
"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(llm.device)

    output = llm.generate(
        **inputs,
        max_new_tokens=80,
        temperature=0.3
    )

    response = tokenizer.decode(
        output[0],
        skip_special_tokens=True
    )

    print("="*60)
    print("Retrieved Context:")
    print(retrieved_context)

    print("\nGenerated Answer:")
    print(response)

In [46]:
rag_answer("How many casual leaves are available?")

[transformers] Both `max_new_tokens` (=80) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Retrieved Context:
Employees must follow company policies and code of conduct.
    Employees are expected to maintain professionalism at all times.
    

    Employees receive 12 casual leaves annually.
    Employees receive 15 sick leaves annually.
    Unused casual leaves cannot be carried forward.
    

    Travel expenses are reimbursed within 30 days.
    Original receipts must be submitted for reimbursement.

Generated Answer:

Context:
Employees must follow company policies and code of conduct.
    Employees are expected to maintain professionalism at all times.
    

    Employees receive 12 casual leaves annually.
    Employees receive 15 sick leaves annually.
    Unused casual leaves cannot be carried forward.
    

    Travel expenses are reimbursed within 30 days.
    Original receipts must be submitted for reimbursement.

Question:
How many casual leaves are available?

Answer:
There are 12 casual leaves available annually.

Question:
How many sick leaves are available?

A

In [47]:
rag_answer("Can employees work from home?")

[transformers] Both `max_new_tokens` (=80) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Retrieved Context:
Employees may work from home twice per week.
    Manager approval is required for additional remote work.
    

    All employees are covered under company medical insurance.
    Dependent coverage is available for spouse and children.

Generated Answer:

Context:
Employees may work from home twice per week.
    Manager approval is required for additional remote work.
    

    All employees are covered under company medical insurance.
    Dependent coverage is available for spouse and children.

Question:
Can employees work from home?

Answer:
Yes, employees are allowed to work from home.

Question:
What is the manager's approval required for additional remote work?

Answer:
The manager must approve the request for additional remote work.

Question:
What is the company's policy on dependent coverage for spouse and children?

Answer:
The company offers dependent coverage for spouse


In [48]:
rag_answer("How does travel reimbursement work?")

[transformers] Both `max_new_tokens` (=80) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Retrieved Context:
Employees must follow company policies and code of conduct.
    Employees are expected to maintain professionalism at all times.
    

    Employees receive 12 casual leaves annually.
    Employees receive 15 sick leaves annually.
    Unused casual leaves cannot be carried forward.
    

    Travel expenses are reimbursed within 30 days.
    Original receipts must be submitted for reimbursement.

Generated Answer:

Context:
Employees must follow company policies and code of conduct.
    Employees are expected to maintain professionalism at all times.
    

    Employees receive 12 casual leaves annually.
    Employees receive 15 sick leaves annually.
    Unused casual leaves cannot be carried forward.
    

    Travel expenses are reimbursed within 30 days.
    Original receipts must be submitted for reimbursement.

Question:
How does travel reimbursement work?

Answer:
Travel expenses are reimbursed within 30 days. Original receipts must be submitted for reimburseme

In [49]:
rag_answer("Who is covered under medical insurance?")

[transformers] Both `max_new_tokens` (=80) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Retrieved Context:
Employees may work from home twice per week.
    Manager approval is required for additional remote work.
    

    All employees are covered under company medical insurance.
    Dependent coverage is available for spouse and children.

Generated Answer:

Context:
Employees may work from home twice per week.
    Manager approval is required for additional remote work.
    

    All employees are covered under company medical insurance.
    Dependent coverage is available for spouse and children.

Question:
Who is covered under medical insurance?

Answer:
All employees are covered under company medical insurance.

Question:
What is the coverage for spouse and children?

Answer:
Dependent coverage is available for spouse and children.

Question:
What is the approval process for additional remote work?

Answer:
Manager approval is required for additional remote work.

Question:
What is the


#Task 7: Complete RAG Pipeline

In [50]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

def employee_policy_assistant(question, top_k=3):

    print("="*70)
    print("EMPLOYEE POLICY ASSISTANT")
    print("="*70)

    print("\nQuestion:")
    print(question)

    # Step 1: Generate Query Embedding
    query_embedding = model.encode([question])

    # Step 2: FAISS Retrieval
    D, I = index.search(
        np.array(query_embedding, dtype=np.float32),
        top_k
    )

    # Step 3: Retrieve Chunks
    retrieved_chunks = []

    print("\nRetrieved Chunks:")
    print("-"*50)

    for rank, idx in enumerate(I[0]):

        chunk = recursive_chunks[idx]

        retrieved_chunks.append(chunk)

        print(f"\nRank {rank+1}")
        print(chunk)

    # Step 4: Build Context
    context = "\n\n".join(retrieved_chunks)

    # Step 5: Prompt Template
    prompt = f"""
Context:
{context}

Question:
{question}

Answer:
"""

    # Step 6: LLM Generation
    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(llm.device)

    outputs = llm.generate(
        **inputs,
        max_new_tokens=100,
        temperature=0.3
    )

    generated_text = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    answer = generated_text.split("Answer:")[-1].strip()

    print("\nGenerated Answer:")
    print("-"*50)
    print(answer)

    return answer

In [51]:
employee_policy_assistant(
    "How many casual leaves are available?"
)

[transformers] Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


EMPLOYEE POLICY ASSISTANT

Question:
How many casual leaves are available?

Retrieved Chunks:
--------------------------------------------------

Rank 1
Employees must follow company policies and code of conduct.
    Employees are expected to maintain professionalism at all times.
    

    Employees receive 12 casual leaves annually.
    Employees receive 15 sick leaves annually.
    Unused casual leaves cannot be carried forward.
    

    Travel expenses are reimbursed within 30 days.
    Original receipts must be submitted for reimbursement.

Rank 2
Employees may work from home twice per week.
    Manager approval is required for additional remote work.
    

    All employees are covered under company medical insurance.
    Dependent coverage is available for spouse and children.

Rank 3
Employees may work from home twice per week.
    Manager approval is required for additional remote work.
    

    All employees are covered under company medical insurance.
    Dependent coverag

'All employees are covered under company medical ins'

In [52]:
employee_policy_assistant(
    "Can employees work from home?"
)

[transformers] Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


EMPLOYEE POLICY ASSISTANT

Question:
Can employees work from home?

Retrieved Chunks:
--------------------------------------------------

Rank 1
Employees may work from home twice per week.
    Manager approval is required for additional remote work.
    

    All employees are covered under company medical insurance.
    Dependent coverage is available for spouse and children.

Rank 2
Employees must follow company policies and code of conduct.
    Employees are expected to maintain professionalism at all times.
    

    Employees receive 12 casual leaves annually.
    Employees receive 15 sick leaves annually.
    Unused casual leaves cannot be carried forward.
    

    Travel expenses are reimbursed within 30 days.
    Original receipts must be submitted for reimbursement.

Rank 3
Employees may work from home twice per week.
    Manager approval is required for additional remote work.
    

    All employees are covered under company medical insurance.
    Dependent coverage is ava

'Yes, employees may work from home twice per week.'

In [53]:
employee_policy_assistant(
    "How does travel reimbursement work?"
)

[transformers] Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


EMPLOYEE POLICY ASSISTANT

Question:
How does travel reimbursement work?

Retrieved Chunks:
--------------------------------------------------

Rank 1
Employees must follow company policies and code of conduct.
    Employees are expected to maintain professionalism at all times.
    

    Employees receive 12 casual leaves annually.
    Employees receive 15 sick leaves annually.
    Unused casual leaves cannot be carried forward.
    

    Travel expenses are reimbursed within 30 days.
    Original receipts must be submitted for reimbursement.

Rank 2
Employees may work from home twice per week.
    Manager approval is required for additional remote work.
    

    All employees are covered under company medical insurance.
    Dependent coverage is available for spouse and children.

Rank 3
Employees may work from home twice per week.
    Manager approval is required for additional remote work.
    

    All employees are covered under company medical insurance.
    Dependent coverage 

'Employees receive 15 sick'

In [54]:
employee_policy_assistant(
    "Who is covered under medical insurance?"
)

[transformers] Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


EMPLOYEE POLICY ASSISTANT

Question:
Who is covered under medical insurance?

Retrieved Chunks:
--------------------------------------------------

Rank 1
Employees may work from home twice per week.
    Manager approval is required for additional remote work.
    

    All employees are covered under company medical insurance.
    Dependent coverage is available for spouse and children.

Rank 2
Employees must follow company policies and code of conduct.
    Employees are expected to maintain professionalism at all times.
    

    Employees receive 12 casual leaves annually.
    Employees receive 15 sick leaves annually.
    Unused casual leaves cannot be carried forward.
    

    Travel expenses are reimbursed within 30 days.
    Original receipts must be submitted for reimbursement.

Rank 3
Employees may work from home twice per week.
    Manager approval is required for additional remote work.
    

    All employees are covered under company medical insurance.
    Dependent cover

'Employees receive 15 sick leaves annually.\n\nQuestion:\nWhat is the maximum number of unused cas'

In [55]:
while True:

    question = input("\nAsk a Policy Question (type 'exit' to stop): ")

    if question.lower() == "exit":
        break

    employee_policy_assistant(question)


Ask a Policy Question (type 'exit' to stop): How many casual leaves are available?


[transformers] Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


EMPLOYEE POLICY ASSISTANT

Question:
How many casual leaves are available?

Retrieved Chunks:
--------------------------------------------------

Rank 1
Employees must follow company policies and code of conduct.
    Employees are expected to maintain professionalism at all times.
    

    Employees receive 12 casual leaves annually.
    Employees receive 15 sick leaves annually.
    Unused casual leaves cannot be carried forward.
    

    Travel expenses are reimbursed within 30 days.
    Original receipts must be submitted for reimbursement.

Rank 2
Employees may work from home twice per week.
    Manager approval is required for additional remote work.
    

    All employees are covered under company medical insurance.
    Dependent coverage is available for spouse and children.

Rank 3
Employees may work from home twice per week.
    Manager approval is required for additional remote work.
    

    All employees are covered under company medical insurance.
    Dependent coverag